# Un ajuste LoRA de verdad

El [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/ajustefino.html) dice tres cosas que aquí se comprueban con código: que LoRA entrena una fracción diminuta del modelo, que lo difícil no es entrenar sino preparar los datos, y que **la comparación honesta es contra el modelo base con un buen prompt**, no contra el modelo base a pelo.

Vamos a enseñarle al asistente el formato de respuesta de la secretaría. Es un caso de libro para ajuste fino: forma, no hechos.

## Aviso sobre el tiempo

Esto entrena de verdad, en CPU y sin GPU. Con un modelo de 0.6B, LoRA y un conjunto pequeño, el entrenamiento tarda menos de un minuto, que es en sí mismo parte de la lección: **ajustar un modelo pequeño ya no es un proyecto**. Lo lento del cuaderno es medir, porque cada evaluación son varias generaciones completas.

## Preparación

In [ ]:
!pip install -q peft duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base_dir = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base_dir = COLAB

sys.path.insert(0, str(base_dir.resolve()))

from secretaria import preparar

ctx = preparar()
con = ctx.conectar()

## Los datos, que es el trabajo de verdad

El capítulo insiste en que entrenar es la parte fácil y que el conjunto de ejemplos es lo que decide el resultado. Aquí tenemos una ventaja poco habitual: **los ejemplos se generan del almacén**, así que son consistentes por construcción.

El formato que queremos enseñar es este, siempre igual:

```
**Trámite:** matrícula ordinaria · **Plazo:** del 2026-07-15 al 2026-07-31 · **Dónde:** sede electrónica
```

Fijaos en la partición. Entrenamos con unos trámites y evaluamos con **otros distintos**, porque si se solapan estaríamos midiendo memoria en lugar de formato.

In [ ]:
import random

FORMATO = ("**Trámite:** {t} · **Plazo:** del {i} al {f} · "
           "**Dónde:** sede electrónica")

filas = con.execute(
    "select tramite, fecha_inicio, fecha_fin from dim_plazo order by tramite"
).fetchall()

random.Random(7).shuffle(filas)
corte = int(len(filas) * 0.7)
ENTRENO, PRUEBA = filas[:corte], filas[corte:]

PREGUNTAS = [
    "¿cuál es el plazo de {t}?",
    "¿hasta cuándo puedo hacer {t}?",
    "necesito saber las fechas de {t}",
]


def ejemplos(filas, semilla=0):
    rnd = random.Random(semilla)
    return [
        {"pregunta": rnd.choice(PREGUNTAS).format(t=t.replace("_", " ")),
         "respuesta": FORMATO.format(t=t.replace("_", " "), i=i, f=f)}
        for t, i, f in filas
    ]


ENTRENAMIENTO = ejemplos(ENTRENO, 1)
EVALUACION = ejemplos(PRUEBA, 2)

print(f"{len(ENTRENAMIENTO)} ejemplos de entrenamiento, {len(EVALUACION)} de evaluación")
print(f"trámites de entrenamiento: {[t for t, _, _ in ENTRENO][:4]} ...")
print(f"trámites de evaluación:    {[t for t, _, _ in PRUEBA]}")
print(f"\nun ejemplo:\n  U: {ENTRENAMIENTO[0]['pregunta']}\n  A: {ENTRENAMIENTO[0]['respuesta']}")

Nueve ejemplos de entrenamiento. Es ridículamente poco, y es a propósito: el capítulo dice que para enseñar formato bastan unos cientos bien hechos, así que aquí vamos un orden de magnitud por debajo para ver hasta dónde llega.

## La alternativa a batir

Antes de entrenar nada hay que medir lo que ya se tiene. El capítulo lo pide explícitamente: la comparación no es contra el modelo desnudo, es contra **el modelo base con un buen prompt**, que es lo que cuesta cero y se despliega en cualquier sitio.

In [ ]:
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)

SISTEMA_SIMPLE = "Eres el asistente de la secretaría académica."

SISTEMA_BUENO = """Eres el asistente de la secretaría académica.
Responde SIEMPRE con esta única línea, sin añadir nada más:
**Trámite:** <nombre> · **Plazo:** del <AAAA-MM-DD> al <AAAA-MM-DD> · **Dónde:** sede electrónica

Ejemplo:
**Trámite:** cambio grupo · **Plazo:** del 2026-09-15 al 2026-09-25 · **Dónde:** sede electrónica"""

PATRON_FORMATO = re.compile(
    r"\*\*Trámite:\*\*.+·\s*\*\*Plazo:\*\*\s*del\s*\d{4}-\d{2}-\d{2}\s*al\s*"
    r"\d{4}-\d{2}-\d{2}\s*·\s*\*\*Dónde:\*\*"
)


def responder(sistema, pregunta, red=None, max_tokens=60):
    mensajes = [{"role": "system", "content": sistema},
                {"role": "user", "content": pregunta}]
    texto = tok.apply_chat_template(mensajes, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        salida = (red or modelo).generate(**entrada, max_new_tokens=max_tokens,
                                          do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True).strip()


def medir(nombre, sistema, red=None):
    aciertos = 0
    for caso in EVALUACION:
        salida = responder(sistema, caso["pregunta"], red)
        aciertos += bool(PATRON_FORMATO.search(salida))
    print(f"  {nombre:34s} formato correcto {aciertos}/{len(EVALUACION)}")
    return aciertos


print("Antes de entrenar nada:")
base_simple = medir("base, prompt escueto", SISTEMA_SIMPLE)
base_bueno = medir("base, prompt con el formato", SISTEMA_BUENO)

Ese segundo número es la vara de medir. Cualquier ajuste fino que no lo supere no ha servido para nada, y esta comparación es justo la que se omite cuando alguien enseña un modelo ajustado y lo compara con el modelo desnudo.

Vedlo también en lo que cuesta: el prompt bueno son seis líneas de texto, cero minutos de entrenamiento y funciona con cualquier proveedor.

## LoRA: el número que lo explica todo

Ahora sí. Configuramos LoRA con los dos parámetros que el capítulo dice que hay que saber declarar: el **rango** y **a qué capas se aplica**.

In [ ]:
from peft import LoraConfig, get_peft_model

configuracion = LoraConfig(
    r=8,                       # el rango: cuánta capacidad tiene la corrección
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # las proyecciones de atención
    task_type="CAUSAL_LM",
)

ajustado = get_peft_model(modelo, configuracion)

entrenables = sum(p.numel() for p in ajustado.parameters() if p.requires_grad)
totales = sum(p.numel() for p in ajustado.parameters())

print(f"parámetros totales:     {totales:,}")
print(f"parámetros entrenables: {entrenables:,}")
print(f"proporción:             {100 * entrenables / totales:.3f} %")

Ese porcentaje es la razón de que esto quepa en un portátil. No estamos moviendo un modelo de 600 millones de parámetros: estamos moviendo dos matrices pequeñas por cada proyección de atención, y el resto está congelado.

Y de ahí sale la otra ventaja que el capítulo menciona: lo que se guarda al final son esos pocos megabytes, no una copia del modelo. Se pueden tener veinte adaptadores para veinte tareas sobre el mismo modelo base, y quitar cualquiera de ellos es borrar un fichero.

## Entrenar

El bucle de entrenamiento más simple que se puede escribir. Sin `Trainer`, sin abstracciones, para que se vea que aquí no hay nada raro: se calcula la pérdida, se propaga y se actualizan los pesos que no están congelados.

**Esta es la celda lenta.** Unos diez minutos en CPU.

In [ ]:
import time


def como_lote(ejemplo):
    """Convierte un par pregunta/respuesta en tokens con sus etiquetas."""
    mensajes = [{"role": "system", "content": SISTEMA_SIMPLE},
                {"role": "user", "content": ejemplo["pregunta"]}]
    prefijo = tok.apply_chat_template(mensajes, tokenize=False,
                                      add_generation_prompt=True, enable_thinking=False)
    completo = prefijo + ejemplo["respuesta"] + tok.eos_token
    ids = tok(completo, return_tensors="pt").input_ids
    etiquetas = ids.clone()
    # No se penaliza al modelo por el prompt, solo por lo que tiene que escribir.
    etiquetas[:, : tok(prefijo, return_tensors="pt").input_ids.shape[1]] = -100
    return ids, etiquetas


optimizador = torch.optim.AdamW(
    [p for p in ajustado.parameters() if p.requires_grad], lr=2e-4
)

ajustado.train()
inicio = time.time()
EPOCAS = 2
perdidas = []

for epoca in range(EPOCAS):
    for i, ejemplo in enumerate(ENTRENAMIENTO):
        ids, etiquetas = como_lote(ejemplo)
        salida = ajustado(input_ids=ids, labels=etiquetas)
        salida.loss.backward()
        optimizador.step()
        optimizador.zero_grad()
        perdidas.append(salida.loss.item())

    print(f"época {epoca + 1}: pérdida media {sum(perdidas[-len(ENTRENAMIENTO):]) / len(ENTRENAMIENTO):.4f}"
          f"   ({time.time() - inicio:.0f}s)")

ajustado.eval()
print(f"\nprimera pérdida: {perdidas[0]:.4f}   última: {perdidas[-1]:.4f}")
print(f"tiempo total: {time.time() - inicio:.0f}s")

## La comparación que importa

Con el adaptador entrenado, medimos sobre los trámites que **no** aparecieron en el entrenamiento. Y lo comparamos con las dos referencias de antes.

In [ ]:
print("Sobre trámites que el modelo no ha visto entrenando:\n")

# OJO: `get_peft_model` envuelve el modelo EN SU SITIO. Después de esto,
# `modelo` ya lleva los adaptadores dentro, así que medirlo no daría el
# modelo base sino el ajustado. Para recuperar el base hay que apagarlos.
with ajustado.disable_adapter():
    medir("base, prompt escueto", SISTEMA_SIMPLE, ajustado)
    medir("base, prompt con el formato", SISTEMA_BUENO, ajustado)

medir("ajustado, prompt escueto", SISTEMA_SIMPLE, ajustado)
medir("ajustado, prompt con el formato", SISTEMA_BUENO, ajustado)

El `disable_adapter()` no es un detalle de implementación: es **la comprobación de que el modelo base sigue intacto**. Apagando el adaptador vuelve a comportarse exactamente como antes de entrenar, porque nunca se le tocó un peso.

Y es también una trampa fácil de pisar. `get_peft_model` envuelve el modelo en su sitio, así que si después del entrenamiento medís la variable original creyendo que es el modelo base, estáis midiendo el ajustado y la comparación no vale nada. Fue justo el error que tuvo este cuaderno mientras se escribía.

In [ ]:
# Un vistazo a lo que produce cada uno con la misma pregunta.
pregunta = EVALUACION[0]["pregunta"]
print(f"P: {pregunta}\n")

with ajustado.disable_adapter():
    print(f"base, prompt escueto:\n  {responder(SISTEMA_SIMPLE, pregunta, ajustado)[:160]}\n")
    print(f"base, prompt bueno:\n  {responder(SISTEMA_BUENO, pregunta, ajustado)[:160]}\n")

print(f"ajustado, prompt escueto:\n  {responder(SISTEMA_SIMPLE, pregunta, ajustado)[:160]}\n")
print(f"lo que debería decir:\n  {EVALUACION[0]['respuesta']}")

Ahí está lo que el número no contaba.

El **base con el prompt bueno** produce la estructura entera y las fechas casi bien. Falla la métrica por una sola cosa: se come los asteriscos del negrita. Nuestro patrón los exige, así que puntúa cero.

El **ajustado** clava el formato, asteriscos incluidos, y dentro escribe *"convocatoria electrónica electrónica"* y unas fechas de 2027 que no existen en ninguna parte.

O sea que el modelo que gana la medición es el que dice cosas falsas con mejor presentación. Vamos a medir también el contenido.

In [ ]:
def fechas_correctas(nombre, sistema, red=None, con_adaptador=True):
    """¿Las fechas que da coinciden con las del almacén?"""
    aciertos = 0
    for caso, (tramite, inicio, fin) in zip(EVALUACION, PRUEBA):
        salida = responder(sistema, caso["pregunta"], red)
        aciertos += (str(inicio) in salida and str(fin) in salida)
    print(f"  {nombre:34s} fechas correctas {aciertos}/{len(EVALUACION)}")
    return aciertos


print("Ahora el contenido, no la forma:\n")
with ajustado.disable_adapter():
    fechas_correctas("base, prompt con el formato", SISTEMA_BUENO, ajustado)
fechas_correctas("ajustado, prompt escueto", SISTEMA_SIMPLE, ajustado)

Ninguno acierta las fechas, y no podía ser de otra manera: **son trámites que ninguno de los dos ha visto nunca**. El ajuste fino no las metió dentro del modelo, y el prompt tampoco se las dio.

Con eso ya se puede decir lo que ha pasado, que es la lección más útil del cuaderno y no la que yo esperaba escribir:

**El ajuste fino optimiza exactamente lo que mediste.** Le pedimos formato, medimos formato, y nos dio formato impecable envolviendo un contenido inventado. La métrica subió de 0 a 4 sobre 5 mientras la utilidad real no se movía de cero.

Es el mismo aviso del [capítulo de evaluación](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/evaluacion.html) visto desde el otro lado. Allí decíamos que un juez sin validar produce métricas decorativas; aquí se ve que **una métrica incompleta produce modelos decorativos**. Si lo único que medís es la forma, entrenar os dará forma.

Y confirma la frase del capítulo, ahora con las dos mitades:

* **Forma**: el ajuste la aprende con nueve ejemplos y veinticinco segundos. Espectacular.
* **Hechos**: no los aprende, y no hay conjunto de entrenamiento razonable que se los enseñe, porque cambian. Para eso están [las herramientas](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/herramientas.html).

La conclusión práctica no es que el ajuste fino no sirva. Es que sirve **para lo que sirve**, y que un sistema útil aquí sería el modelo ajustado para la forma **más** una llamada al almacén para las fechas. Ninguna de las dos piezas sobra y ninguna sustituye a la otra.

El ajuste gana, y por goleada. Con nueve ejemplos y veinticinco segundos de entrenamiento, el modelo ajustado produce el formato casi siempre y el base no lo produce nunca.

Antes de celebrarlo, mirad **qué** produce cada uno.

## Lo que cuesta guardarlo

Ya que está entrenado, midamos lo que ocupa. Es el argumento práctico de LoRA y se ve mejor con el número delante.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as carpeta:
    ajustado.save_pretrained(carpeta)
    tamano = sum(
        os.path.getsize(os.path.join(raiz, f))
        for raiz, _, ficheros in os.walk(carpeta) for f in ficheros
    )
    ficheros = [f for _, _, fs in os.walk(carpeta) for f in fs]

print(f"el adaptador ocupa {tamano / 1e6:.1f} MB")
print(f"ficheros: {ficheros}")
print(f"\nel modelo base ocupa del orden de {totales * 4 / 1e9:.1f} GB en float32")
print(f"o sea, el adaptador es un {100 * tamano / (totales * 4):.2f} % del modelo")

Un puñado de megabytes contra varios gigabytes. Eso es lo que permite tener un adaptador por tarea, por cliente o por idioma sin multiplicar el almacenamiento, y lo que hace que revertir sea borrar un fichero.

Ahora bien, recordad el aviso del capítulo sobre [lo que os deja atado](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/ajustefino.html): ese fichero solo sirve con **este** modelo base y con un proveedor que acepte servirlo. En el momento en que salga un modelo mejor, el adaptador no se traslada: hay que volver a entrenar.

## Ejercicios

**1. Subid el rango.** Repetid con `r=32` y comparad formato acertado, parámetros entrenables, tiempo y tamaño del adaptador. ¿Compensa?

**2. Más ejemplos.** Generad ejemplos también de `dim_asignatura` para triplicar el conjunto y volved a medir. El capítulo dice que unos cientos bastan para formato: comprobad dónde deja de mejorar.

**3. Ensuciad el conjunto.** Cambiad el formato en un tercio de los ejemplos de entrenamiento y medid. Es el aviso sobre consistencia, y el resultado suele ser peor de lo que uno espera.

**4. ¿Y los hechos?** Preguntad al modelo ajustado por un trámite de entrenamiento y comprobad si acierta la fecha. Después por uno de evaluación. La diferencia entre las dos respuestas es toda la tesis del capítulo.

**5. Contra un buen prompt de verdad.** Mejorad `SISTEMA_BUENO` con dos ejemplos más y volved a medir. Si el prompt alcanza al ajuste, tenéis la respuesta a si hacía falta entrenar.

**6. Olvido catastrófico.** Preguntad al modelo ajustado algo que no tenga nada que ver con plazos y ved si sigue contestando con el formato. Un ajuste demasiado agresivo estropea lo que el modelo ya sabía hacer.

## Lo que os lleváis

* **LoRA entrena una fracción diminuta**, menos del medio por ciento aquí, y por eso esto cabe en un portátil.
* **El adaptador son unos megabytes**, no una copia del modelo. Uno por tarea, y quitarlo es borrar un fichero.
* **Lo difícil son los datos.** El bucle de entrenamiento cabe en veinte líneas; el conjunto de ejemplos decide el resultado.
* **Enseña forma, no hechos.** Aprende el formato en veinticinco segundos y sigue inventándose las fechas que no vio. Lo útil es ajuste **más** herramienta, no ajuste en lugar de herramienta.
* **La comparación honesta es contra el modelo base con un buen prompt**, no contra el modelo desnudo. Es la que casi nunca se enseña.
* **El ajuste optimiza exactamente lo que midáis.** Aquí subió el formato de 0 a 4 sobre 5 sin mejorar ni un ápice el contenido. Una métrica incompleta produce modelos decorativos.
* **`get_peft_model` envuelve el modelo en su sitio.** Si medís la variable original creyendo que es el base, estáis midiendo el ajustado. Se apaga con `disable_adapter()`.
* **Evaluad con casos que no estuvieran en el entrenamiento.** Si se solapan, estáis midiendo memoria.

Con esto cerramos lo que se le puede hacer a un modelo por dentro. La siguiente parte va de lo que le metemos: [el contexto](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/intro.html).